In [1]:
import pandas as pd
from ml_utils.src import *
from ml_utils.config import *

import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df_train = pd.read_parquet(DATA_DIR, columns = colunas)
df_test = pd.read_parquet("2023", columns = colunas)

In [3]:
df_train = preparar_dados(df_train, objetivo = '', n_samples = 100_000)
df_test = preparar_dados(df_test, objetivo = '', n_samples = 10_000)

In [4]:
X_train = df_train.drop(['FALTOU'], axis=1)
y_train = df_train['FALTOU']

X_test = df_test.drop(['FALTOU'], axis=1)
y_test = df_test['FALTOU']

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
clf = RandomForestClassifier()

clf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, clf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  clf.predict(X_val))))

print(classification_report(y_val, clf.predict(X_val)))

Ein:  0.0176
Eval: 0.3441
              precision    recall  f1-score   support

           0       0.70      0.78      0.74     12243
           1       0.56      0.46      0.51      7652

    accuracy                           0.66     19895
   macro avg       0.63      0.62      0.62     19895
weighted avg       0.65      0.66      0.65     19895



In [7]:
cv_rf = buscar_hiperparametros_rf(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomForestClassifier(class_weight='balanced', max_depth=20,
                       max_features='log2', max_samples=1.0,
                       min_samples_leaf=20, min_samples_split=50,
                       random_state=42)
Ein:  0.3086
Eout: 0.3377
              precision    recall  f1-score   support

           0       0.77      0.65      0.70     12243
           1       0.55      0.68      0.61      7652

    accuracy                           0.66     19895
   macro avg       0.66      0.67      0.66     19895
weighted avg       0.68      0.66      0.67     19895



In [8]:
rf = RandomForestClassifier(
    **cv_rf.best_params_,
    class_weight='balanced',  
    random_state=42
)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

#joblib.dump(rf, 'rf_presenca.pkl')

Ein:  0.3086
Eout: 0.3463
              precision    recall  f1-score   support

           0       0.79      0.68      0.73      6815
           1       0.47      0.61      0.53      3185

    accuracy                           0.65     10000
   macro avg       0.63      0.64      0.63     10000
weighted avg       0.68      0.65      0.66     10000



In [ ]:
def tune_random_forest(x_train, y_train, n_iter=10, cv=5, scoring='f1_weighted', random_state=42):
    
    n_estimators = [int(x) for x in np.linspace(start=50, stop=200, num=10)]
    max_features = ['sqrt', 'log2']
    max_depth = [int(x) for x in np.linspace(start=10, stop=40, num=4)]
    max_depth.append(None)
    
    param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_features':      ['sqrt', 'log2', 0.3],  # adiciona fração
    'max_depth':         [5, 10, 15, 20],         # valores menores
    'min_samples_split': [20, 50, 100],            # mais restritivo
    'min_samples_leaf':  [20, 50, 100],            # mais restritivo
    'max_samples':       [0.6, 0.8, 1.0],         # subsampling de linhas
}

    rf = RandomForestClassifier(class_weight='balanced', random_state=random_state)
    
    cv_rf = RandomizedSearchCV(      # ← mudou
        estimator=rf,
        param_distributions=param_grid,  # ← mudou
        n_iter=n_iter,               # ← agora usa o parâmetro
        cv=cv,
        scoring=scoring,
        verbose=2,
        n_jobs=-1,
        random_state=random_state    # ← reprodutibilidade
    )

    cv_rf.fit(x_train, y_train)

    return cv_rf


cv_rf = tune_random_forest(X_train, y_train, n_iter=30, cv=5, scoring='f1_weighted', random_state=42)

print(cv_rf.best_estimator_)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))